In [1]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

print("Loading data...")
heart = fetch_ucirepo(id=45)
X = heart.data.features.copy()
y = heart.data.targets.copy()

y = y['num'].apply(lambda v: 1 if v > 0 else 0)

X = X.apply(pd.to_numeric, errors='coerce')
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

print("Rows:", X.shape[0], "| Columns:", X.shape[1])
print("NaN remaining:", X.isnull().sum().sum())
print("y distribution:\n", y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training model...")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n========== RESULTS ==========")
print(classification_report(y_test, y_pred,
    target_names=["No Disease", "Disease"]))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.3f}")

Loading data...
Rows: 303 | Columns: 13
NaN remaining: 0
y distribution:
 num
0    164
1    139
Name: count, dtype: int64
Training model...

========== RESULTS ==========
              precision    recall  f1-score   support

  No Disease       0.96      0.82      0.89        33
     Disease       0.82      0.96      0.89        28

    accuracy                           0.89        61
   macro avg       0.89      0.89      0.89        61
weighted avg       0.90      0.89      0.89        61

ROC-AUC Score: 0.951


In [2]:
import pickle

# Save the trained model
with open('heart_disease_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("✅ Model saved as heart_disease_model.pkl")

# Save the scaler too (important!)
with open('heart_disease_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Scaler saved as heart_disease_scaler.pkl")

✅ Model saved as heart_disease_model.pkl
✅ Scaler saved as heart_disease_scaler.pkl
